In [21]:


import sys
from pathlib import Path
import joblib

project_root = Path(r"d:/projects/CreditPathAI_Oct_Batch/kaggle_dataset")
sys.path.append(str(project_root))

print("Project root added:", project_root)

import pandas as pd

from src.preprocess import *


Project root added: d:\projects\CreditPathAI_Oct_Batch\kaggle_dataset


In [22]:

data_path = r"D:/projects/CreditPathAI_Oct_Batch/kaggle_dataset/data/interim/loan_data_clean.csv"
preprocessed_path = r"D:/projects/CreditPathAI_Oct_Batch/kaggle_dataset/data/processed/loan_data_preprocessed.csv"
df = pd.read_csv(data_path)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Missing values:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("Duplicate rows:", df.duplicated().sum())


Shape: (38480, 37)
Columns: ['Unnamed: 0', 'id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'purpose', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'inq_last_6mths', 'mths_since_last_delinq', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'repay_fail']
Missing values:
 loan_amnt                     1
funded_amnt                   1
funded_amnt_inv               1
installment                   1
emp_length                  993
annual_inc                    2
delinq_2yrs                   1
inq_last_6mths                1
mths_since_last_delinq    24363
open_acc                      1
pub_rec                       1
revol_bal                     4
revol_util       

In [23]:
cols_to_drop = [col for col in df.columns if df[col].isnull().sum() == len(df)]
df.drop(columns=cols_to_drop, inplace=True)
print("Dropped Columns:", cols_to_drop)

Dropped Columns: []


In [24]:


# GENERIC PROCESSING
df = handle_missing_values(df)
df = convert_percent_to_float(df, percent_cols=["int_rate"])
df = one_hot_encode(df)
#df = handle_outliers_iqr(df, method="cap")
#df, scaler = scale_features(df, method="standard")




In [25]:
convert_floats_to_float32(df)
downcast_integers(df)

Downcasting 19 float64 columns to float32.
Downcasting 4 int64 columns to smaller integer types.


,Unnamed: 0,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,dti,...,last_credit_pull_d_Oct-15,last_credit_pull_d_Sep-07,last_credit_pull_d_Sep-08,last_credit_pull_d_Sep-09,last_credit_pull_d_Sep-10,last_credit_pull_d_Sep-11,last_credit_pull_d_Sep-12,last_credit_pull_d_Sep-13,last_credit_pull_d_Sep-14,last_credit_pull_d_Sep-15
0,2,2,2,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,...,False,False,False,False,False,False,False,False,False,False
1,3,545583,703644,2500.0,2500.0,2500.000000,13.980000,85.419998,20004.0,19.860001,...,False,False,False,False,False,False,False,False,False,False
2,4,532101,687836,5000.0,5000.0,5000.000000,15.950000,175.669998,59000.0,19.570000,...,False,False,False,False,False,False,False,False,False,False
3,5,877788,1092507,7000.0,7000.0,7000.000000,9.910000,225.580002,53796.0,10.800000,...,False,False,False,False,False,False,False,False,False,False
4,6,875406,1089981,2000.0,2000.0,2000.000000,5.420000,60.320000,30000.0,3.600000,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38475,38476,849205,1060907,3000.0,3000.0,3000.000000,11.990000,99.629997,74250.0,22.170000,...,False,False,False,False,False,False,False,False,False,False
38476,38477,852914,1065048,10400.0,10400.0,10400.000000,13.490000,352.880005,62000.0,11.570000,...,False,False,False,False,False,False,False,False,False,False
38477,38478,519553,671637,16000.0,10550.0,10531.358398,14.960000,250.770004,95088.0,10.630000,...,False,False,False,False,False,False,False,False,False,False
38478,38479,825638,1034448,10000.0,10000.0,10000.000000,16.889999,355.989990,48720.0,18.969999,...,False,False,False,False,False,False,False,False,False,True


In [26]:
target_col = "repay_fail"

X = df.drop(columns=[target_col])
y = df[target_col]


In [27]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [28]:
X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns,
    index=X.index
)

df_processed = pd.concat([X_scaled, y], axis=1)


In [29]:
# Save the preprocessed data
df.to_csv(preprocessed_path, index=False)
print("Preprocessed data saved to:", preprocessed_path)

Preprocessed data saved to: D:/projects/CreditPathAI_Oct_Batch/kaggle_dataset/data/processed/loan_data_preprocessed.csv


In [30]:
artifacts = {
    "scaler": scaler,
    "feature_order": X.columns.tolist()
}

joblib.dump(
     artifacts,r"D:/projects/CreditPathAI_Oct_Batch/kaggle_dataset/artifacts/preprocess_artifacts.joblib"
    
)


['D:/projects/CreditPathAI_Oct_Batch/kaggle_dataset/artifacts/preprocess_artifacts.joblib']

In [31]:
print(df["repay_fail"].value_counts())


repay_fail
0    32651
1     5829
Name: count, dtype: int64


In [32]:
print(df["repay_fail"].value_counts())


repay_fail
0    32651
1     5829
Name: count, dtype: int64


In [33]:
assert y.nunique() == 2
